# **Fake News Detection using DistilBERT**

## **Fine-Tuning DistilBERT**

While SBERT provides general language understanding, Fine-Tuning a model like DistilBERT allows the model's internal weights to actually "adapt" to the specific patterns of our fake news dataset.

**Key Decisions:**

- **Model**: `distilbert-base-uncased`. It retains 97% of BERT's performance while being 40% smaller and 60% faster.

- **Max Length** (64): News headlines are short. Standard BERT uses 512 tokens, but reducing this to 64 significantly saves GPU memory and speeds up training without losing information.

- **Weight Decay** (0.01): Added as a regularization technique to prevent the model from memorizing the training set (overfitting).

## **0. Upload Files**

This notebook is designed to run on **Google Colab** with a free T4 GPU, which is all you need.

Before running any code, upload your two data files:
- `training_data.csv` — labeled headlines (tab-separated, label in first column: `0` = fake, `1` = real)
- `testing_data.csv` — unlabeled headlines (label column contains `2` as a placeholder)

You can upload via the **Colab file manager** (folder icon in the left sidebar) or by running the upload cell below.

In [1]:
from google.colab import files

print("Please upload 'training_data.csv':")
uploaded_train = files.upload()

print("Please upload 'testing_data.csv':")
uploaded_test = files.upload()

# This will save the uploaded files to the content directory, which is what your current code expects (e.g., '/training_data.csv')

Please upload 'training_data.csv':


Saving training_data.csv to training_data.csv
Please upload 'testing_data.csv':


Saving testing_data.csv to testing_data.csv


## **1. Install Necessary Libraries**

We install and import only what we need to keep the environment clean:

| Library | Purpose |
|---|---|
| `transformers` | Provides the pre-trained DistilBERT model, tokenizer, `Trainer`, and `TrainingArguments` |
| `datasets` | HuggingFace utility for dataset handling (pulled in as a dependency) |
| `accelerate` | Required by HuggingFace `Trainer` for efficient GPU training |
| `torch` | PyTorch — the deep learning backend that runs the model on the GPU |
| `sklearn` | Used only for `train_test_split` and `accuracy_score` — no ML model from sklearn here |

> **Note:** `transformers[torch]` installs the PyTorch-compatible version of the library automatically.

In [2]:
!pip install -q transformers[torch] datasets accelerate

import pandas as pd
import numpy as np
import torch
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

## **2. Configuration & Device**

All key settings are centralised here so you can change them in one place without touching the rest of the code.

| Variable | Value | Why |
|---|---|---|
| `MODEL_NAME` | `distilbert-base-uncased` | 40% smaller than BERT, 60% faster, retains 97% of performance. `uncased` means it lowercases input — appropriate since our cleaning step already lowercases headlines. |
| `TRAIN_PATH` | `training_data.csv` | Tab-separated, no header, 34k rows |
| `TEST_PATH` | `testing_data.csv` | Tab-separated, no header, ~10k rows — labels are `2` (placeholder) |
| `DEVICE` | auto-detected | `cuda` if a GPU is available (T4 on Colab), otherwise `cpu`. Training on CPU is possible but ~20× slower. |


If Colab shows `Using device: cuda`, you're on GPU — training will take ~7 minutes. If it shows `cpu`, go to **Runtime → Change runtime type → T4 GPU**.

In [3]:
MODEL_NAME = "distilbert-base-uncased"
TRAIN_PATH = 'training_data.csv'
TEST_PATH  = 'training_data.csv'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


## **3. Data Presprocessing**

### **3.1. Data Preprocessing Function**

Even though DistilBERT is a powerful contextual model, a small amount of text cleaning still helps:

- **Encoding fix:** The CSV contains smart quotes and en-dashes encoded as garbled bytes (e.g. `‚` instead of `'`). These become noise tokens that the tokenizer maps to `[UNK]`. We strip them.
- **Lowercase:** We force lowercase to match the `uncased` checkpoint — DistilBERT's vocabulary was built on lowercased text, so this ensures consistent tokenization.
- **Whitespace collapse:** Multiple spaces and newlines are normalised to a single space. This prevents the tokenizer from seeing meaningless padding as separate tokens.

We do **not** remove stop words or punctuation — DistilBERT's attention mechanism extracts meaning from full sentences, including function words and punctuation.

In [4]:
def clean(text: str) -> str:
    text = str(text).encode('utf-8', errors='ignore').decode('utf-8')
    text = re.sub(r'[\x80-\x9f\u201c\u201d\u2018\u2019\u2013\u2014]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

### **3.2. Load & Prepare Data**



We split the training data into a **90% train / 10% validation** set using `train_test_split` with `random_state=42` for reproducibility.

The validation set is used during training to:
1. Monitor for overfitting (if val loss goes up while train loss goes down, we're overfitting)
2. Select the best checkpoint via `load_best_model_at_end=True` in `TrainingArguments`

The test set (`testing_data.csv`) is kept completely separate — it's only used at the very end to generate the final predictions, never during training or model selection.

In [5]:
try:
    train_df = pd.read_csv(TRAIN_PATH, sep='\t', header=None, names=['label', 'headline'])
    test_df = pd.read_csv(TEST_PATH, sep='\t', header=None, names=['label', 'headline'])

    train_df['clean'] = train_df['headline'].fillna('').apply(clean)
    test_df['clean'] = test_df['headline'].fillna('').apply(clean)
except FileNotFoundError:
    print("Error: Please upload 'training_data.csv' and 'testing_data.csv'.")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['clean'].tolist(), train_df['label'].tolist(), test_size=0.1, random_state=42
)

## **4. Tokenization**

The tokenizer converts raw text strings into the numerical format DistilBERT expects: token IDs, attention masks, and (optionally) token type IDs.

**Key choices:**

- **`max_length=64`:** News headlines are short — the longest in this dataset is under 50 words. Standard BERT uses 512 tokens, but that would waste ~8× memory and compute. 64 is safely above the max headline length in our data.
- **`truncation=True`:** Silently truncates any headline that somehow exceeds 64 tokens. Safety net only — in practice this never triggers here.
- **`padding=True`:** Pads shorter sequences to the same length within each batch (batch-level padding, not global). This is required because PyTorch tensors in a batch must have the same shape.

The tokenizer is loaded from the same checkpoint as the model (`distilbert-base-uncased`) to guarantee the vocabulary is consistent.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=64)
val_encodings   = tokenizer(val_texts, truncation=True, padding=True, max_length=64)
test_encodings  = tokenizer(test_df['clean'].tolist(), truncation=True, padding=True, max_length=64)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## **5. PyTorch DataSet & Class**

HuggingFace's `Trainer` expects data as a PyTorch `Dataset` object — a class that implements `__getitem__` and `__len__`.

`FakeNewsDataset` is a thin wrapper that:
1. Stores the tokenizer output (`encodings`) and the labels
2. Returns a dictionary of tensors for each sample — exactly what the `Trainer`'s internal DataLoader needs
3. Reports its length so the DataLoader knows how many batches to create

Each call to `__getitem__` converts one sample's encoding values to `torch.tensor` on the fly. The `Trainer` handles batching, shuffling, and moving tensors to the GPU automatically.

In [7]:
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FakeNewsDataset(train_encodings, train_labels)
val_dataset   = FakeNewsDataset(val_encodings, val_labels)

## **6. Model & Training Set-Up**

**Model loading:**
`AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)` loads the pre-trained DistilBERT weights and **replaces the original language-modelling head with a fresh 2-class classification head**. This is why the load report shows `MISSING` keys for `classifier.*` and `pre_classifier.*` — those layers are randomly initialized and will be learned during fine-tuning.

**Key `TrainingArguments`:**

| Argument | Value | Reasoning |
|---|---|---|
| `num_train_epochs` | 2 | More epochs risk overfitting on a task this well-separated. Val accuracy plateaued at epoch 2 (98.19%). |
| `per_device_train_batch_size` | 32 | Fits comfortably in T4 GPU VRAM (16GB) with `max_length=64`. |
| `warmup_steps` | 100 | Gradually ramps learning rate from 0 to avoid large gradient updates on the randomly-initialized classification head early in training. |
| `weight_decay` | 0.1 | L2 regularisation applied to all non-bias weights. Reduces overfitting. |
| `eval_strategy` | `"epoch"` | Validates after every epoch so we can track progress and catch overfitting early. |
| `load_best_model_at_end` | `True` | Automatically restores the best checkpoint (lowest val loss) after training finishes. |
| `report_to` | `"none"` | Disables Weights & Biases / TensorBoard logging to avoid noisy auth prompts on Colab. |

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,                # REDUCED epochs to 2 to prevent late-stage overfitting
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.1,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## **7. Execution, Evalutaion & Save Test Presictions**

This cell runs the entire fine-tuning loop and saves the predictions in one go.

**Training (`trainer.train()`):**
The `Trainer` handles the full training loop: gradient computation, backpropagation, learning rate scheduling, validation after each epoch, and checkpoint saving. On a Colab T4 GPU this takes roughly **7 minutes** for 2 epochs over 30k training samples.

**Prediction (`trainer.predict()`):**
We pass the test set with dummy labels (`[0]*len(test_df)`) — the actual labels are ignored during prediction. `argmax` over the two logit columns picks the winning class per headline.

**Output file:**
`testing_data_predicted_distilBERT.csv` is saved in the **original format** (tab-separated, no header, `label\theadline`) with placeholder `2` labels replaced by `0` (fake) or `1` (real). Download it from the Colab file sidebar.

In [9]:
print("\n--- Starting Fine-Tuning ---")
trainer.train()

# EVALUATE AND SAVE TEST PREDICTIONS
print("\n--- Generating Test Predictions ---")
raw_preds = trainer.predict(FakeNewsDataset(test_encodings, [0]*len(test_df)))
final_preds = np.argmax(raw_preds.predictions, axis=1)

# Save to CSV in the original format
output = test_df[['headline']].copy()
output.insert(0, 'label', final_preds)
output.to_csv('testing_data_predicted_distilBERT.csv', sep='\t', header=False, index=False)

print("Done! Download 'testing_data_predicted_distilBERT.csv' from the sidebar.")


--- Starting Fine-Tuning ---


Epoch,Training Loss,Validation Loss,Accuracy
1,0.047961,0.064963,0.977752
2,0.020839,0.075188,0.980972


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



--- Generating Test Predictions ---


Done! Download 'testing_data_predicted_distilBERT.csv' from the sidebar.
